In [ ]:
import sys
sys.path.insert(0, "/Users/nautilus/cobblestone_study")
import pandas as pd
from config import CLEAN_PARQUET, OUTPUTS_DIR
# sys.path.insert(0, str(Path(__file__).parent.parent))



In [6]:
df = pd.read_parquet("/Users/nautilus/cobblestone_study/data/clean.parquet")
df.head()

,prices,load_fc,wind_fc,solar_fc,actual_load,actual_gen,residual_load_fc
2024-06-12 00:00:00+02:00,88.85,43510.4775,10215.6250,0.00,45384.5300,39235.5900,33294.8525
2024-06-12 01:00:00+02:00,85.35,42501.1275,10056.7950,0.00,43697.0700,38589.1400,32444.3325
2024-06-12 02:00:00+02:00,80.29,42080.3675,9831.2350,0.00,43011.9625,39225.4450,32249.1325
2024-06-12 03:00:00+02:00,82.11,42441.8100,9400.2350,0.00,43002.1425,38758.1925,33041.5750
2024-06-12 04:00:00+02:00,86.11,44073.2475,8982.6375,1.16,44276.2100,38214.9025,35089.4500


In [20]:
# Check time frame has no gap
expected = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h', tz="Europe/Berlin")
missing = expected.difference(df.index)
extra = df.index.difference(expected)

print(f"Expected hours: {len(expected)}")
print(f"Actual hours: {len(df)}")
print(f"Missing hours: {len(missing)}")
print(f"Extra hours: {len(extra)}")

if len(missing) > 0:
    print(missing[:10])

Expected hours: 17521
Actual hours: 17521
Missing hours: 0
Extra hours: 0


In [21]:
# Check time fram has no duplicate
duplicates = df.index.duplicated().sum()
print(f"Duplicate timestamps: {duplicates}")

Duplicate timestamps: 0


In [23]:
# NaN couns
nan_counts = df.isnull().sum()
nan_pct = (df.isnull().mean() * 100).round(2)

nan_report = pd.DataFrame({
    "missing_count" : nan_counts,
    "missing_pct" : nan_pct
})

print(nan_report)

                  missing_count  missing_pct
prices                        1         0.01
load_fc                       2         0.01
wind_fc                       3         0.02
solar_fc                      3         0.02
actual_load                   7         0.04
actual_gen                 8767        50.04
residual_load_fc              3         0.02


In [ ]:
# Price Sanity check
prices = df["prices"].dropna()

print(f"Min price:  {prices.min():.2f} Euro/MWh")   
# Negative Price : Not a bug, renewable energy, low demand!
print(f"Max price:  {prices.max():.2f} Euro/MWh")
print(f"Mean price: {prices.mean():.2f} Euro/MWh")
print(f"Negative prices: {(prices < 0).sum()}")
print(f"Extreme spikes (>500): {(prices > 500).sum()}")

# flatline (same price 5+ hours...?)
flatline = (prices == prices.shift(1)) & (prices == prices.shift(2)) & \
    (prices == prices.shift(3)) & (prices == prices.shift(4))
print(f"Flatline periods (5h+): {flatline.sum()}")

Min price:  -499.00 Euro/MWh
Max price:  936.28 Euro/MWh
Mean price: 90.60 Euro/MWh
Negative prices: 1112
Extreme spikes (>500): 16
Flatline periods (5h+): 0


In [ ]:
# Day Light Saving
daily_hours = df.groupby(df.index.date).size()
non_24 = daily_hours[daily_hours != 24]
print(f"Days with != 24 hours: {len(non_24)}")
print(non_24)

Days with != 24 hours: 5
2024-10-27    25
2025-03-30    23
2025-10-26    25
2026-03-29    23
2026-06-12     1
dtype: int64


In [31]:
# 6. Coverage Table:
coverage = pd.DataFrame({
    "start" : df.apply(lambda col : col.first_valid_index()),
    "end" : df.apply(lambda col: col.last_valid_index()),
    "total_rows": len(df),
    "non_null": df.count(),
    "pct_complete": (df.count() / len(df) * 100).round(2)
})

print(coverage)

                                     start                       end  \
prices           2024-06-12 00:00:00+02:00 2026-06-12 00:00:00+02:00   
load_fc          2024-06-12 00:00:00+02:00 2026-06-11 23:00:00+02:00   
wind_fc          2024-06-12 00:00:00+02:00 2026-06-11 23:00:00+02:00   
solar_fc         2024-06-12 00:00:00+02:00 2026-06-11 23:00:00+02:00   
actual_load      2024-06-12 00:00:00+02:00 2026-06-11 17:00:00+02:00   
actual_gen       2024-06-12 00:00:00+02:00 2026-06-11 17:00:00+02:00   
residual_load_fc 2024-06-12 00:00:00+02:00 2026-06-11 23:00:00+02:00   

                  total_rows  non_null  pct_complete  
prices                 17521     17520         99.99  
load_fc                17521     17519         99.99  
wind_fc                17521     17518         99.98  
solar_fc               17521     17518         99.98  
actual_load            17521     17514         99.96  
actual_gen             17521      8754         49.96  
residual_load_fc       17521     17518